In [ ]:
# Exploratory scVI and scANVI label-transfer sandbox.
import os
import tempfile

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scrublet as scr
# [Note added for repository release] scrublet is imported but never called in this notebook.
import scvi
import seaborn as sns
import torch

In [ ]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

In [ ]:
sc.set_figure_params(scanpy=True, dpi=300, dpi_save=300, 
                         frameon=True, vector_friendly=True, fontsize=10, figsize= (5,5), color_map=None, format='pdf', 
                         facecolor=None, transparent=False, ipython_format='png2x')

In [ ]:
heartmomac = sc.read('/home/yanjun2/scanpy/atlas/heartmomac.h5ad')

In [ ]:
heartmomac

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
sc.pl.umap(heartmomac, color = ['cell_type'])

In [ ]:
adata.obs['cell_type'] = adata.obs['leiden_res_0_8']
adata.obs['sample_id'] = adata.obs['hash.ID']

In [ ]:
adata = adata.concatenate(heartmomac, join='inner'
                         )

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color = ['cell_type', 'sample_id'])

In [ ]:
import scanpy.external as sce
sce.pp.harmony_integrate(adata, 'sample_id')

In [ ]:
adata

In [ ]:
sc.pl.pca(adata, color = ['cell_type'], legend_loc = 'on data')

In [ ]:
adata_bbknn = adata.copy()
sc.pp.pca(adata_bbknn)

In [ ]:
import bbknn


In [ ]:
# Neighbours, UMAP and clustering
sc.pp.neighbors(adata_bbknn)

In [ ]:
sc.tl.umap(adata_bbknn)
sc.pl.umap(adata_bbknn, color=['cell_type'], wspace=1)

In [ ]:
sc.pl.umap(adata_bbknn, color=['cell_type'], wspace=1, legend_loc = 'on data')

In [ ]:
adata = sc.read('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
sc.pl.umap(momacverse, color = ['Clusters', 'Patients'])

In [ ]:
print(momacverse.X)

In [ ]:
momacverse

In [ ]:
# scVI integration
scvi.model.SCVI.setup_anndata(momacverse, batch_key="Patients")

In [ ]:
scvi_ref = scvi.model.SCVI(
    momacverse,
    use_layer_norm="none",
    use_batch_norm="none",
    encode_covariates=True,
    dropout_rate=0.2,
    n_layers=2,
)
scvi_ref.train()


In [ ]:
SCVI_LATENT_KEY = "X_scVI"

momacverse.obsm[SCVI_LATENT_KEY] = scvi_ref.get_latent_representation()
sc.pp.neighbors(momacverse, use_rep=SCVI_LATENT_KEY)
sc.tl.leiden(momacverse)
sc.tl.umap(momacverse)

In [ ]:
sc.pl.umap(
    momacverse,
    color=["Patients", "Clusters"],
    frameon=False,
    ncols=1,
)

In [ ]:
ref_adata_path = os.path.join(save_dir.name, "momacverse.h5ad")

In [ ]:
scvi_ref_path = os.path.join(save_dir.name, "momacverse")
scvi_ref.save(scvi_ref_path, overwrite=True)

In [ ]:
# again a no-op in this tutorial, but good practice to use
scvi.model.SCANVI.prepare_query_anndata(adata, momacverse)

In [ ]:
scanvi_query = scvi.model.SCANVI.load_query_data(adata, momacverse)

In [ ]:
scanvi_query.train(
    max_epochs=100,
    plan_kwargs={"weight_decay": 0.0},
    check_val_every_n_epoch=10,
)

In [ ]:
SCANVI_PREDICTIONS_KEY = "predictions_scanvi"

pancreas_query.obsm[SCANVI_LATENT_KEY] = scanvi_query.get_latent_representation()
pancreas_query.obs[SCANVI_PREDICTIONS_KEY] = scanvi_query.predict()

In [ ]:
df = pancreas_query.obs.groupby(["celltype", SCANVI_PREDICTIONS_KEY]).size().unstack(fill_value=0)
norm_df = df / df.sum(axis=0)

plt.figure(figsize=(8, 8))
_ = plt.pcolor(norm_df)
_ = plt.xticks(np.arange(0.5, len(df.columns), 1), df.columns, rotation=90)
_ = plt.yticks(np.arange(0.5, len(df.index), 1), df.index)
plt.xlabel("Predicted")
plt.ylabel("Observed")


In [ ]:
pancreas_full = anndata.concat([pancreas_query, pancreas_ref], label="batch")
pancreas_full

In [ ]:
pancreas_full.obs["batch"] = pancreas_full.obs["batch"].cat.rename_categories(
    ["Query", "Reference"]
)

In [ ]:
full_predictions = scanvi_query.predict(pancreas_full)
print(f"Acc: {np.mean(full_predictions == pancreas_full.obs['celltype'])}")

pancreas_full.obs[SCANVI_PREDICTIONS_KEY] = full_predictions